|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Tensor parallelism<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: two shardings, one collective<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import torch

torch.manual_seed(0)
D, H, RANKS = 256, 1024, 4
x  = torch.randn(8, D)
W1 = torch.randn(D, H)/D**0.5
W2 = torch.randn(H, D)/H**0.5
reference = torch.relu(x @ W1) @ W2
print('reference', tuple(reference.shape))

Shard an MLP across four ranks two different ways, both correct, and count
what each one costs in conversation.

All on the CPU. Stage 20 does it with real collectives; this is about the
design rule, which is the part you have to get right before the collectives
exist.

# Exercise 1: column, then row

In [ ]:
def mlp_parallel(x, W1, W2, ranks):
  W1s = list(W1.chunk(ranks, dim=1))       # COLUMNS
  W2s = list(W2.chunk(ranks, dim=0))       # ROWS, matching
  partials = [torch.relu(x @ W1s[r]) @ W2s[r] for r in range(ranks)]
  return sum(partials), 1                  # one all-reduce

out, collectives = mlp_parallel(x, W1, W2, RANKS)
print(f'max difference {(out-reference).abs().max().item():.2e}')
print(f'collectives per block: {collectives}')

# Exercise 2: the other way round

Also correct. Count the collectives.

In [ ]:
def mlp_wrong(x, W1, W2, ranks):
  """Row-parallel first. Now the hidden activations must be gathered
  BEFORE relu, because each rank holds a partial sum of them."""
  W1s = list(W1.chunk(ranks, dim=0))       # rows: shards the INPUT
  xs  = list(x.chunk(ranks, dim=1))
  h = sum(xs[r] @ W1s[r] for r in range(ranks))   # collective 1
  h = torch.relu(h)
  W2s = list(W2.chunk(ranks, dim=0))
  hs  = list(h.chunk(ranks, dim=1))
  out = sum(hs[r] @ W2s[r] for r in range(ranks)) # collective 2
  return out, 2

out2, c2 = mlp_wrong(x, W1, W2, RANKS)
print(f'max difference {(out2-reference).abs().max().item():.2e}   (still correct)')
print(f'collectives per block: {c2}   <- twice the talking, same answer')

# Exercise 3: what a collective costs

A ring all-reduce moves `2(R-1)/R` times the tensor. Put that next to a
decode step.

In [ ]:
def bytes_per_step(batch, d_model, ranks, layers=32, collectives=1):
  return 2 * batch * d_model * 2 * (ranks-1)/ranks * layers * collectives

PCIE, NVLINK, STEP_MS = 12e9, 300e9, 10.0
print(f"{'batch':>6} {'1 collective':>14} {'2 collectives':>15}  (PCIe, % of step)")
for b in (1, 32, 256):
  one = 1000*bytes_per_step(b, 4096, 8, collectives=1)/PCIE
  two = 1000*bytes_per_step(b, 4096, 8, collectives=2)/PCIE
  print(f'{b:>6} {100*one/STEP_MS:>13.1f}% {100*two/STEP_MS:>14.1f}%')

### Both arrangements are correct. Only one is usable.

Column-then-row needs one all-reduce per block. Row-then-column needs
two, because `relu` is not linear and a partial sum cannot be passed
through it.

That is the whole design rule, and it generalises: **put the collective
where the non-linearity is not**. Attention splits the same way for the
same reason, whole heads per rank, because softmax reduces over a row
and a rank must own the whole row.

### And the number that decides whether any of this is worth it

At batch 1 the collectives are a rounding error even on PCIe. At batch
256 with eight ranks they are most of the step, and doubling them takes
you past it.

So tensor parallelism and continuous batching pull against each other.
Every plot in Part 2 said to raise the batch; this one says the
interconnect gets a vote. On NVLink it does not care. On PCIe it decides
your maximum batch size, which decides your throughput, which was the
thing you split the model to get.

Stage 20 builds it with real collectives and checks that the two-rank
output matches the one-rank output exactly. The lesson is where the
communication lands, not the speed: on one GPU there is none to be had.

    ./vc guide 20